In [ ]:
import sqlite3
from pathlib import Path

# Create a sqlite db file in the Colab runtime (or use ':memory:' for in-memory)
db_path = Path("warehouse.db")
conn = sqlite3.connect(db_path)
cur = conn.cursor()

schema_sql = r"""
-- Paste your FULL script here exactly as provided

CREATE TABLE location_file (
  location_id     TEXT PRIMARY KEY,
  zone            TEXT,
  aisle           TEXT,
  bay             TEXT,
  level           TEXT,
  position        TEXT,
  location_type   TEXT,              -- PICK / RESERVE / DOCK / QA / STAGE
  is_active       INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE item_file (
  item_id         TEXT PRIMARY KEY,
  item_desc       TEXT NOT NULL,
  uom             TEXT NOT NULL,      -- EA / CS etc.
  case_pack_qty   INTEGER,
  each_per_case   INTEGER,
  is_active       INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE dealer_file (
  dealer_id       TEXT PRIMARY KEY,
  dealer_name     TEXT NOT NULL,
  address1        TEXT,
  city            TEXT,
  state           TEXT,
  zip             TEXT,
  phone           TEXT,
  is_active       INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE warehouse_user_file (
  user_id         TEXT PRIMARY KEY,
  username        TEXT UNIQUE NOT NULL,
  full_name       TEXT,
  role            TEXT NOT NULL,      -- ADMIN / SUPERVISOR / OPERATOR
  is_active       INTEGER NOT NULL DEFAULT 1,
  created_ts      TEXT NOT NULL DEFAULT (datetime('now'))
);

-- Insert sample data
INSERT INTO location_file (location_id, zone, aisle, bay, level, position, location_type)
VALUES
  ('PICK-A01-01-01', 'PICK', 'A01', '01', '01', '01', 'PICK'),
  ('PICK-A01-01-02', 'PICK', 'A01', '01', '01', '02', 'PICK'),
  ('RES-B10-05-01',  'RES',  'B10', '05', '01', '01', 'RESERVE'),
  ('DOCK-01',        'DOCK', NULL,  NULL, NULL, NULL, 'DOCK');

INSERT INTO item_file (item_id, item_desc, uom, case_pack_qty, each_per_case)
VALUES
  ('SKU-1001', 'Oil Filter - Small', 'EA', 12, 12),
  ('SKU-1002', 'Brake Pads - Front', 'EA',  8,  8),
  ('SKU-2001', 'Wiper Blade 18in',   'EA', 24, 24);

INSERT INTO dealer_file (dealer_id, dealer_name, address1, city, state, zip, phone)
VALUES
  ('D-001', 'Northside Auto', '123 Main St', 'Chicago', 'IL', '60601', '312-555-0101'),
  ('D-002', 'Lakeshore Parts', '55 Lake Ave', 'Evanston', 'IL', '60201', '847-555-0199');

INSERT INTO warehouse_user_file (user_id, username, full_name, role)
VALUES
  ('U-001', 'jsmith', 'John Smith', 'OPERATOR'),
  ('U-002', 'mgarcia', 'Maria Garcia', 'SUPERVISOR'),
  ('U-003', 'admin', 'System Admin', 'ADMIN');

CREATE TABLE inventory_file (
  inventory_id    INTEGER PRIMARY KEY AUTOINCREMENT,
  location_id     TEXT NOT NULL,
  item_id         TEXT NOT NULL,

  on_hand_qty     DECIMAL(18,3) NOT NULL DEFAULT 0,
  allocated_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  available_qty  DECIMAL(18,3) NOT NULL DEFAULT 0,

  last_updated_ts TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

  FOREIGN KEY (location_id) REFERENCES location_file(location_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id),

  CONSTRAINT ck_inventory_qty
    CHECK (on_hand_qty >= 0 AND allocated_qty >= 0 AND available_qty >= 0)
);

CREATE UNIQUE INDEX ux_inventory_loc_item
ON inventory_file(location_id, item_id);

CREATE TABLE order_header (
  order_id        TEXT PRIMARY KEY,
  dealer_id       TEXT NOT NULL,

  order_date      DATE NOT NULL,
  status          TEXT NOT NULL
                   CHECK (status IN ('OPEN','ALLOCATED','PICKED','SHIPPED','CANCELLED')),

  priority        INTEGER DEFAULT 5,
  created_by      TEXT,
  created_ts      TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

  FOREIGN KEY (dealer_id) REFERENCES dealer_file(dealer_id),
  FOREIGN KEY (created_by) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE order_detail (
  order_line_id   INTEGER PRIMARY KEY AUTOINCREMENT,
  order_id        TEXT NOT NULL,
  item_id         TEXT NOT NULL,

  ordered_qty     DECIMAL(18,3) NOT NULL,
  allocated_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  picked_qty      DECIMAL(18,3) NOT NULL DEFAULT 0,
  shipped_qty     DECIMAL(18,3) NOT NULL DEFAULT 0,

  FOREIGN KEY (order_id) REFERENCES order_header(order_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id),

  CONSTRAINT ck_order_qty
    CHECK (
      ordered_qty >= 0 AND
      allocated_qty >= 0 AND
      picked_qty >= 0 AND
      shipped_qty >= 0
    )
);

CREATE INDEX ix_order_detail_order ON order_detail(order_id);
CREATE INDEX ix_order_detail_item ON order_detail(item_id);

CREATE TABLE work_assignment_header (
  wa_id            TEXT PRIMARY KEY,

  wa_type          TEXT NOT NULL
                    CHECK (wa_type IN ('PICK','PUTAWAY','REPLEN','COUNT')),

  status           TEXT NOT NULL
                    CHECK (status IN ('NEW','ASSIGNED','IN_PROGRESS','COMPLETED','CLOSED')),

  created_by       TEXT NOT NULL,
  assigned_to      TEXT,
  created_ts       TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  started_ts       TIMESTAMP,
  completed_ts    TIMESTAMP,

  FOREIGN KEY (created_by) REFERENCES warehouse_user_file(user_id),
  FOREIGN KEY (assigned_to) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE work_assignment_detail (
  wa_line_id       INTEGER PRIMARY KEY AUTOINCREMENT,
  wa_id            TEXT NOT NULL,

  task_type        TEXT NOT NULL
                    CHECK (task_type IN ('PICK','PUTAWAY','MOVE','COUNT')),

  item_id          TEXT NOT NULL,
  from_location_id TEXT,
  to_location_id   TEXT,

  requested_qty    DECIMAL(18,3) NOT NULL,
  confirmed_qty    DECIMAL(18,3),

  source_order_id  TEXT,
  source_order_line_id INTEGER,

  status           TEXT NOT NULL
                    CHECK (status IN ('OPEN','CONFIRMED','CANCELLED')),

  confirmed_by     TEXT,
  confirmed_ts     TIMESTAMP,

  FOREIGN KEY (wa_id) REFERENCES work_assignment_header(wa_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id),
  FOREIGN KEY (from_location_id) REFERENCES location_file(location_id),
  FOREIGN KEY (to_location_id) REFERENCES location_file(location_id),
  FOREIGN KEY (confirmed_by) REFERENCES warehouse_user_file(user_id),
  FOREIGN KEY (source_order_id) REFERENCES order_header(order_id)
);

CREATE INDEX ix_wa_detail_wa ON work_assignment_detail(wa_id);

CREATE TABLE case_header (
  case_id                 TEXT PRIMARY KEY,
  container_id            TEXT,

  status                  TEXT NOT NULL
                           CHECK (status IN ('EXPECTED','RECEIVED','RECONCILED','CANCELLED')),

  expected_date           DATE NOT NULL,
  case_received_date      DATE,
  reconciliation_due_date DATE NOT NULL,
  reconciliation_date     DATE,

  reconciled_by           TEXT,  -- user_id OR 'SYSTEM'

  created_ts              TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  updated_ts              TIMESTAMP,

  CONSTRAINT ck_reconciled_by_valid
    CHECK (reconciled_by IS NULL OR reconciled_by = 'SYSTEM' OR LENGTH(reconciled_by) > 0)
);

CREATE TABLE case_detail (
  case_line_id     INTEGER PRIMARY KEY AUTOINCREMENT,
  case_id          TEXT NOT NULL,
  item_id          TEXT NOT NULL,

  qty_to_receive   DECIMAL(18,3) NOT NULL,
  received_qty     DECIMAL(18,3) NOT NULL DEFAULT 0,

  FOREIGN KEY (case_id) REFERENCES case_header(case_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id),

  CONSTRAINT ck_case_qty_nonnegative
    CHECK (qty_to_receive >= 0 AND received_qty >= 0),

  CONSTRAINT ck_case_received_not_exceed
    CHECK (received_qty <= qty_to_receive)
);

CREATE INDEX ix_case_detail_case ON case_detail(case_id);
CREATE INDEX ix_case_detail_item ON case_detail(item_id);

CREATE TABLE invoice_header (
  invoice_id      TEXT PRIMARY KEY,
  order_id        TEXT NOT NULL,
  dealer_id       TEXT NOT NULL,

  invoice_date    DATE NOT NULL,
  status          TEXT NOT NULL
                   CHECK (status IN ('OPEN','POSTED','CANCELLED')),

  total_amount    DECIMAL(18,2),

  created_by      TEXT,
  created_ts      TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

  FOREIGN KEY (order_id) REFERENCES order_header(order_id),
  FOREIGN KEY (dealer_id) REFERENCES dealer_file(dealer_id),
  FOREIGN KEY (created_by) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE invoice_detail (
  invoice_line_id INTEGER PRIMARY KEY AUTOINCREMENT,
  invoice_id      TEXT NOT NULL,
  item_id         TEXT NOT NULL,

  qty             DECIMAL(18,3) NOT NULL,
  unit_price      DECIMAL(18,2) NOT NULL,
  ext_amount      DECIMAL(18,2) NOT NULL,

  FOREIGN KEY (invoice_id) REFERENCES invoice_header(invoice_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id)
);

CREATE INDEX ix_invoice_detail_invoice ON invoice_detail(invoice_id);
"""

# Recommended for SQLite: enable FK checks (OFF by default unless set per-connection)
cur.execute("PRAGMA foreign_keys = ON;")

# Run full script
cur.executescript(schema_sql)
conn.commit()

# Quick verification: list tables
tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()
print("Tables created:")
for (t,) in tables:
    print(" -", t)

conn.close()

print("\nDB saved at:", db_path.resolve())

Tables created:
 - case_detail
 - case_header
 - dealer_file
 - inventory_file
 - invoice_detail
 - invoice_header
 - item_file
 - location_file
 - order_detail
 - order_header
 - sqlite_sequence
 - warehouse_user_file
 - work_assignment_detail
 - work_assignment_header

DB saved at: /content/warehouse.db


In [ ]:
import sqlite3
conn = sqlite3.connect("warehouse.db")
cur = conn.cursor()

rows = cur.execute("""
SELECT l.location_id, l.location_type
FROM location_file l
ORDER BY l.location_id;
""").fetchall()

rows

[('DOCK-01', 'DOCK'),
 ('PICK-A01-01-01', 'PICK'),
 ('PICK-A01-01-02', 'PICK'),
 ('RES-B10-05-01', 'RESERVE')]

In [ ]:
import sqlite3, random
from datetime import datetime, date, timedelta

DB_NAME = "warehouse.db"

# ----------------------------
# CONFIG (change these)
# ----------------------------
CFG = {
    "clear_existing": True,   # True = wipe tables then insert fresh synthetic data

    "n_locations": 120,
    "n_items": 250,
    "n_dealers": 80,
    "n_users": 30,

    "n_cases": 300,
    "case_lines_min": 1,
    "case_lines_max": 6,

    "n_orders": 600,
    "order_lines_min": 1,
    "order_lines_max": 8,

    "prob_case_received": 0.75,
    "prob_reconcile_system": 0.80,

    "prob_order_cancelled": 0.03,

    "inventory_locations_sample": 100,  # locations to stock
    "inventory_items_sample": 200,      # items to stock per selected location (capped by n_items)
}

random.seed(42)  # reproducible


# ----------------------------
# Helpers
# ----------------------------
def d_to_str(d):
    return d.strftime("%Y-%m-%d") if d else None

def dt_to_str(dt):
    return dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None

def rand_time_dt(base_date: date, hour_lo=6, hour_hi=20):
    return datetime.combine(base_date, datetime.min.time()).replace(
        hour=random.randint(hour_lo, hour_hi),
        minute=random.randint(0, 59),
        second=random.randint(0, 59),
    )

def gen_id(prefix: str, i: int, width=4) -> str:
    return f"{prefix}-{i:0{width}d}"

def pick_some(seq, k):
    k = min(k, len(seq))
    return random.sample(seq, k=k)


# ----------------------------
# Seed
# ----------------------------
conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

if CFG["clear_existing"]:
    # delete in FK-safe order
    cur.executescript("""
    DELETE FROM invoice_detail;
    DELETE FROM invoice_header;

    DELETE FROM work_assignment_detail;
    DELETE FROM work_assignment_header;

    DELETE FROM order_detail;
    DELETE FROM order_header;

    DELETE FROM case_detail;
    DELETE FROM case_header;

    DELETE FROM inventory_file;

    DELETE FROM location_file;
    DELETE FROM item_file;
    DELETE FROM dealer_file;
    DELETE FROM warehouse_user_file;
    """)
    conn.commit()

today = datetime.now().date()

# ----------------------------
# 1) Master tables
# ----------------------------
# Locations
locations = []
for i in range(1, CFG["n_locations"] + 1):
    if i <= 5:
        loc_id = f"DOCK-{i:02d}"
        locations.append((loc_id, "DOCK", None, None, None, None, "DOCK", 1))
    elif i <= 15:
        loc_id = f"STAGE-{i-5:02d}"
        locations.append((loc_id, "STAGE", None, None, None, None, "STAGE", 1))
    else:
        zone = random.choice(["PICK", "RES"])
        aisle = f"{random.choice(list('ABCDEFGH'))}{random.randint(1, 20):02d}"
        bay = f"{random.randint(1, 30):02d}"
        level = f"{random.randint(1, 5):02d}"
        pos = f"{random.randint(1, 6):02d}"
        loc_type = "PICK" if zone == "PICK" else "RESERVE"
        loc_id = f"{loc_type}-{aisle}-{bay}-{level}-{pos}"
        locations.append((loc_id, zone, aisle, bay, level, pos, loc_type, 1))

cur.executemany("""
INSERT INTO location_file
(location_id, zone, aisle, bay, level, position, location_type, is_active)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", locations)

# Items
items = []
nouns = ["Filter", "Brake Pad", "Wiper Blade", "Spark Plug", "Belt", "Hose", "Bearing", "Sensor", "Rotor", "Battery"]
sizes = ["Small", "Medium", "Large", "XL", "18in", "20in", "24in", "Front", "Rear"]
for i in range(1, CFG["n_items"] + 1):
    item_id = f"SKU-{1000+i}"
    desc = f"{random.choice(nouns)} - {random.choice(sizes)}"
    case_pack = random.choice([6, 8, 10, 12, 16, 24, 36])
    items.append((item_id, desc, "EA", case_pack, case_pack, 1))

cur.executemany("""
INSERT INTO item_file
(item_id, item_desc, uom, case_pack_qty, each_per_case, is_active)
VALUES (?, ?, ?, ?, ?, ?)
""", items)

# Dealers
dealers = []
cities = [("Chicago", "IL", "60601"), ("Evanston", "IL", "60201"), ("Aurora", "IL", "60505"),
          ("Naperville", "IL", "60540"), ("Joliet", "IL", "60431"), ("Elgin", "IL", "60120")]
for i in range(1, CFG["n_dealers"] + 1):
    dealer_id = gen_id("D", i, width=3)
    city, st, zp = random.choice(cities)
    dealers.append((
        dealer_id,
        f"Dealer {i:03d}",
        f"{random.randint(10,9999)} {random.choice(['Main','Lake','Oak','Pine','Market'])} St",
        city, st, zp,
        f"{random.randint(200,999)}-555-{random.randint(1000,9999)}",
        1
    ))

cur.executemany("""
INSERT INTO dealer_file
(dealer_id, dealer_name, address1, city, state, zip, phone, is_active)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", dealers)

# Users
users = []
roles = ["OPERATOR", "SUPERVISOR", "ADMIN"]
for i in range(1, CFG["n_users"] + 1):
    user_id = gen_id("U", i, width=3)
    username = f"user{i:03d}"
    full_name = f"User {i:03d}"
    role = random.choices(roles, weights=[0.75, 0.2, 0.05], k=1)[0]
    users.append((user_id, username, full_name, role, 1))

cur.executemany("""
INSERT INTO warehouse_user_file (user_id, username, full_name, role, is_active)
VALUES (?, ?, ?, ?, ?)
""", users)

conn.commit()

# Load master IDs
LOCATION_IDS = [r[0] for r in cur.execute("SELECT location_id FROM location_file WHERE is_active=1")]
ITEM_IDS     = [r[0] for r in cur.execute("SELECT item_id FROM item_file WHERE is_active=1")]
DEALER_IDS   = [r[0] for r in cur.execute("SELECT dealer_id FROM dealer_file WHERE is_active=1")]
USER_IDS     = [r[0] for r in cur.execute("SELECT user_id FROM warehouse_user_file WHERE is_active=1")]

# ----------------------------
# 2) Inventory
# ----------------------------
inv_rows = []
inv_loc_sample = pick_some(LOCATION_IDS, CFG["inventory_locations_sample"])
for loc in inv_loc_sample:
    item_sample = pick_some(ITEM_IDS, min(CFG["inventory_items_sample"], len(ITEM_IDS)))
    for item in item_sample:
        on_hand = random.randint(0, 400)
        allocated = random.randint(0, min(30, on_hand)) if on_hand > 0 else 0
        available = on_hand - allocated
        inv_rows.append((loc, item, float(on_hand), float(allocated), float(available)))

cur.executemany("""
INSERT OR IGNORE INTO inventory_file
(location_id, item_id, on_hand_qty, allocated_qty, available_qty)
VALUES (?, ?, ?, ?, ?)
""", inv_rows)
conn.commit()

# ----------------------------
# 3) Cases (header/detail)
# ----------------------------
case_header_rows = []
case_detail_rows = []

for i in range(1, CFG["n_cases"] + 1):
    case_id = gen_id("CASE", i, width=4)
    container_id = f"CONT-{random.randint(1, max(1, CFG['n_cases']//10))}"

    expected_date = today - timedelta(days=random.randint(1, 140))
    created_ts = rand_time_dt(expected_date - timedelta(days=5))

    received = random.random() < CFG["prob_case_received"]
    case_received_date = expected_date + timedelta(days=random.randint(0, 3)) if received else None

    due = (case_received_date + timedelta(days=7)) if case_received_date else (expected_date + timedelta(days=21))

    if due <= today:
        reconciled = random.random() < 0.9
    else:
        reconciled = random.random() < 0.1

    if reconciled:
        status = "RECONCILED"
        max_date = min(today, due + timedelta(days=7))
        rec_date = due + timedelta(days=random.randint(0, max(0, (max_date - due).days)))
        reconciled_by = "SYSTEM" if random.random() < CFG["prob_reconcile_system"] else random.choice(USER_IDS)
        updated_ts = rand_time_dt(rec_date)
    else:
        status = "RECEIVED" if case_received_date else "EXPECTED"
        rec_date = None
        reconciled_by = None
        updated_ts = None

    case_header_rows.append((
        case_id, container_id, status,
        d_to_str(expected_date),
        d_to_str(case_received_date),
        d_to_str(due),
        d_to_str(rec_date),
        reconciled_by,
        dt_to_str(created_ts),
        dt_to_str(updated_ts),
    ))

    n_lines = random.randint(CFG["case_lines_min"], CFG["case_lines_max"])
    for item_id in pick_some(ITEM_IDS, n_lines):
        qty_to_receive = round(random.uniform(5, 120), 3)
        if case_received_date:
            received_qty = qty_to_receive if random.random() < 0.85 else round(qty_to_receive * random.uniform(0.5, 0.99), 3)
        else:
            received_qty = round(qty_to_receive * random.uniform(0.0, 0.25), 3)
        received_qty = min(received_qty, qty_to_receive)
        case_detail_rows.append((case_id, item_id, qty_to_receive, received_qty))

cur.executemany("""
INSERT INTO case_header
(case_id, container_id, status, expected_date, case_received_date,
 reconciliation_due_date, reconciliation_date, reconciled_by,
 created_ts, updated_ts)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", case_header_rows)

cur.executemany("""
INSERT INTO case_detail (case_id, item_id, qty_to_receive, received_qty)
VALUES (?, ?, ?, ?)
""", case_detail_rows)
conn.commit()

# ----------------------------
# 4) Orders (header/detail)
# ----------------------------
order_header_rows = []
order_detail_rows = []
statuses = ["OPEN", "ALLOCATED", "PICKED", "SHIPPED"]

for i in range(1, CFG["n_orders"] + 1):
    order_id = gen_id("ORD", i, width=5)
    dealer_id = random.choice(DEALER_IDS)
    created_by = random.choice(USER_IDS)
    order_date = today - timedelta(days=random.randint(1, 90))

    if random.random() < CFG["prob_order_cancelled"]:
        status = "CANCELLED"
    else:
        status = random.choices(statuses, weights=[0.35, 0.25, 0.20, 0.20], k=1)[0]

    priority = random.randint(1, 10)
    order_header_rows.append((order_id, dealer_id, d_to_str(order_date), status, priority, created_by))

    n_lines = random.randint(CFG["order_lines_min"], CFG["order_lines_max"])
    for item_id in pick_some(ITEM_IDS, n_lines):
        qty = round(random.uniform(1, 20), 3)
        order_detail_rows.append((order_id, item_id, qty))

cur.executemany("""
INSERT INTO order_header
(order_id, dealer_id, order_date, status, priority, created_by)
VALUES (?, ?, ?, ?, ?, ?)
""", order_header_rows)

cur.executemany("""
INSERT INTO order_detail (order_id, item_id, ordered_qty)
VALUES (?, ?, ?)
""", order_detail_rows)
conn.commit()

# Map order lines (for work tasks)
order_to_lines = {}
for line_id, oid, item_id, qty in cur.execute("SELECT order_line_id, order_id, item_id, ordered_qty FROM order_detail"):
    order_to_lines.setdefault(oid, []).append((line_id, item_id, qty))

# ----------------------------
# 5) Work assignments
# ----------------------------
wa_header_rows = []
wa_detail_rows = []
wa_counter = 1

orders = list(cur.execute("SELECT order_id, status FROM order_header"))
stage_locs = [l for l in LOCATION_IDS if l.startswith("STAGE-")]

for order_id, order_status in orders:
    if order_status == "CANCELLED":
        continue

    wa_id = gen_id("WA", wa_counter, width=5)
    wa_counter += 1

    created_by = random.choice(USER_IDS)
    assigned_to = random.choice(USER_IDS)

    wa_status = random.choices(
        ["NEW", "ASSIGNED", "IN_PROGRESS", "COMPLETED", "CLOSED"],
        weights=[0.05, 0.20, 0.30, 0.30, 0.15],
        k=1
    )[0]

    wa_header_rows.append((wa_id, "PICK", wa_status, created_by, assigned_to))

    for line_id, item_id, qty in order_to_lines.get(order_id, []):
        from_loc = random.choice(inv_loc_sample) if inv_loc_sample else random.choice(LOCATION_IDS)
        to_loc = random.choice(stage_locs) if stage_locs else None

        detail_status = "CONFIRMED" if wa_status in ("COMPLETED", "CLOSED") and random.random() < 0.9 else "OPEN"
        confirmed_qty = qty if detail_status == "CONFIRMED" else None
        confirmed_by = assigned_to if detail_status == "CONFIRMED" else None
        confirmed_ts = dt_to_str(rand_time_dt(today - timedelta(days=random.randint(0, 30)))) if detail_status == "CONFIRMED" else None

        wa_detail_rows.append((
            wa_id,
            "PICK",
            item_id,
            from_loc,
            to_loc,
            float(qty),
            float(confirmed_qty) if confirmed_qty is not None else None,
            order_id,
            int(line_id),
            detail_status,
            confirmed_by,
            confirmed_ts
        ))

cur.executemany("""
INSERT INTO work_assignment_header
(wa_id, wa_type, status, created_by, assigned_to)
VALUES (?, ?, ?, ?, ?)
""", wa_header_rows)

cur.executemany("""
INSERT INTO work_assignment_detail
(wa_id, task_type, item_id, from_location_id, to_location_id,
 requested_qty, confirmed_qty,
 source_order_id, source_order_line_id,
 status, confirmed_by, confirmed_ts)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", wa_detail_rows)
conn.commit()

# ----------------------------
# 6) Invoices (for shipped orders)
# ----------------------------
invoice_header_rows = []
invoice_detail_rows = []
inv_counter = 1

for order_id, dealer_id, status in cur.execute("SELECT order_id, dealer_id, status FROM order_header"):
    if status != "SHIPPED":
        continue
    # optional skip some
    if random.random() > 0.95:
        continue

    invoice_id = gen_id("INV", inv_counter, width=5)
    inv_counter += 1

    invoice_date = today - timedelta(days=random.randint(0, 20))
    created_by = random.choice(USER_IDS)
    inv_status = random.choices(["OPEN", "POSTED"], weights=[0.25, 0.75], k=1)[0]

    total = 0.0
    for line_id, item_id, qty in order_to_lines.get(order_id, []):
        unit_price = round(random.uniform(5, 120), 2)
        ext = round(float(qty) * unit_price, 2)
        total += ext
        invoice_detail_rows.append((invoice_id, item_id, float(qty), unit_price, ext))

    invoice_header_rows.append((invoice_id, order_id, dealer_id, d_to_str(invoice_date), inv_status, round(total, 2), created_by))

cur.executemany("""
INSERT INTO invoice_header
(invoice_id, order_id, dealer_id, invoice_date, status, total_amount, created_by)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", invoice_header_rows)

cur.executemany("""
INSERT INTO invoice_detail
(invoice_id, item_id, qty, unit_price, ext_amount)
VALUES (?, ?, ?, ?, ?)
""", invoice_detail_rows)
conn.commit()

# ----------------------------
# Summary counts
# ----------------------------
def count(tbl):
    return cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]

print("✅ Synthetic data inserted. Row counts:")
for t in [
    "location_file", "item_file", "dealer_file", "warehouse_user_file",
    "inventory_file",
    "case_header", "case_detail",
    "order_header", "order_detail",
    "work_assignment_header", "work_assignment_detail",
    "invoice_header", "invoice_detail"
]:
    print(f"  {t:24s} {count(t)}")

conn.close()

✅ Synthetic data inserted. Row counts:
  location_file            120
  item_file                250
  dealer_file              80
  warehouse_user_file      30
  inventory_file           20000
  case_header              300
  case_detail              1061
  order_header             600
  order_detail             2650
  work_assignment_header   585
  work_assignment_detail   2585
  invoice_header           104
  invoice_detail           472


In [ ]:
import sqlite3
import pandas as pd

DB_NAME = "warehouse.db"

conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON;")

# Show top 20 rows from case_header
df = pd.read_sql_query("""
SELECT
  case_id,
  container_id,
  status,
  expected_date,
  case_received_date,
  reconciliation_due_date,
  reconciliation_date,
  reconciled_by,
  created_ts,
  updated_ts
FROM case_header
ORDER BY created_ts DESC
LIMIT 20;
""", conn)

conn.close()

df

,case_id,container_id,status,expected_date,case_received_date,reconciliation_due_date,reconciliation_date,reconciled_by,created_ts,updated_ts
0,CASE-0108,CONT-5,RECONCILED,2026-02-19,None,2026-03-12,2026-03-12,U-013,2026-02-14 10:54:35,2026-03-12 18:43:21
1,CASE-0168,CONT-24,RECEIVED,2026-02-19,2026-02-20,2026-02-27,None,None,2026-02-14 07:04:11,None
2,CASE-0175,CONT-2,EXPECTED,2026-02-19,None,2026-03-12,None,None,2026-02-14 06:09:25,None
3,CASE-0104,CONT-12,RECEIVED,2026-02-18,2026-02-20,2026-02-27,None,None,2026-02-13 19:50:01,None
4,CASE-0190,CONT-27,RECEIVED,2026-02-18,2026-02-20,2026-02-27,None,None,2026-02-13 19:37:06,None
5,CASE-0065,CONT-11,RECEIVED,2026-02-18,2026-02-18,2026-02-25,None,None,2026-02-13 16:28:03,None
6,CASE-0035,CONT-18,EXPECTED,2026-02-18,None,2026-03-11,None,None,2026-02-13 15:34:58,None
7,CASE-0279,CONT-25,RECEIVED,2026-02-18,2026-02-21,2026-02-28,None,None,2026-02-13 10:44:27,None
8,CASE-0192,CONT-1,RECEIVED,2026-02-17,2026-02-17,2026-02-24,None,None,2026-02-12 11:37:34,None
9,CASE-0151,CONT-8,EXPECTED,2026-02-17,None,2026-03-10,None,None,2026-02-12 09:09:47,None


#STEP to create a DDL

In [ ]:
import os
EMBED_MODEL = os.getenv("EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
print("Using embedding model:", EMBED_MODEL)

Using embedding model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
!pip -q install sentence-transformers chromadb faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [ ]:
import re

def extract_table_chunks(schema_sql: str):
    pattern = re.compile(
        r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*\((.*?)\)\s*;",
        re.IGNORECASE | re.DOTALL
    )
    matches = pattern.findall(schema_sql)

    chunks = []
    for table, body in matches:
        body_clean = re.sub(r"\s+", " ", body).strip()

        raw_parts = [p.strip() for p in body.split(",") if p.strip()]
        col_names, constraints = [], []
        for part in raw_parts:
            part_clean = re.sub(r"\s+", " ", part).strip()
            if re.match(r"^(CONSTRAINT|FOREIGN KEY|PRIMARY KEY|CHECK|UNIQUE)\b", part_clean, flags=re.I):
                constraints.append(part_clean)
            else:
                first = part_clean.split(" ", 1)[0].strip('"`[]')
                if re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", first):
                    col_names.append(first)

        text = (
            f"Table: {table}. "
            f"Columns: {', '.join(col_names)}. "
            f"Definition: {body_clean}"
        )
        chunks.append({"table": table, "text": text})

    return chunks

table_chunks = extract_table_chunks(schema_sql)
texts = [c["text"] for c in table_chunks]

print("Chunks:", len(table_chunks))
print(texts[0][:250], "...")

Chunks: 13
Table: location_file. Columns: location_id, zone, aisle, bay, level, position, location_type. Definition: location_id TEXT PRIMARY KEY, zone TEXT, aisle TEXT, bay TEXT, level TEXT, position TEXT, location_type TEXT, -- PICK / RESERVE / DOCK / QA / ST ...


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(EMBED_MODEL)

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # good for cosine similarity
)

embeddings = np.array(embeddings, dtype="float32")
print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (13, 384)


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_wms_free")
collection = chroma_client.get_or_create_collection(name="wms_schema_free")

ids = [f"table::{c['table']}" for c in table_chunks]
metadatas = [{"table": c["table"], "type": "ddl_table_chunk"} for c in table_chunks]

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),   # Chroma expects python lists
    metadatas=metadatas
)

print("✅ Stored in Chroma:", collection.count())

✅ Stored in Chroma: 13


In [ ]:
query = "invoice tables and line details"
q_emb = model.encode([query], normalize_embeddings=True).tolist()

res = collection.query(query_embeddings=q_emb, n_results=5)

for i in range(len(res["ids"][0])):
    print("\nID:", res["ids"][0][i])
    print("Table:", res["metadatas"][0][i]["table"])
    print("Doc:", res["documents"][0][i][:220], "...")


ID: table::invoice_detail
Table: invoice_detail
Doc: Table: invoice_detail. Columns: invoice_line_id, invoice_id, item_id, qty, unit_price, ext_amount. Definition: invoice_line_id INTEGER PRIMARY KEY AUTOINCREMENT, invoice_id TEXT NOT NULL, item_id TEXT NOT NULL, qty DECIM ...

ID: table::invoice_header
Table: invoice_header
Doc: Table: invoice_header. Columns: invoice_id, order_id, dealer_id, invoice_date, status, total_amount, created_by, created_ts. Definition: invoice_id TEXT PRIMARY KEY, order_id TEXT NOT NULL, dealer_id TEXT NOT NULL, invoi ...

ID: table::order_detail
Table: order_detail
Doc: Table: order_detail. Columns: order_line_id, order_id, item_id, ordered_qty, allocated_qty, picked_qty, shipped_qty. Definition: order_line_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id TEXT NOT NULL, item_id TEXT NOT N ...

ID: table::case_detail
Table: case_detail
Doc: Table: case_detail. Columns: case_line_id, case_id, item_id, qty_to_receive, received_qty. Definition: case_line_id INT

In [ ]:
# # import faiss

# # X = embeddings.copy()
# # d = X.shape[1]

# # index = faiss.IndexFlatIP(d)  # inner product (works like cosine if normalized)
# # index.add(X)

# # id_lookup = [c["table"] for c in table_chunks]
# # print("✅ FAISS vectors indexed:", index.ntotal)

# def faiss_search(query: str, k: int = 5):
#     q = model.encode([query], normalize_embeddings=True).astype("float32")
#     scores, idxs = index.search(q, k)
#     for score, idx in zip(scores[0], idxs[0]):
#         print(f"score={float(score):.4f}  table={id_lookup[int(idx)]}")

# faiss_search("orders status and priority")

#Steps to Create a Text_SQL

In [ ]:
!pip -q install chromadb sentence-transformers transformers accelerate bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_PATH = "./chroma_wms_free"         # change if you used a different path
COLLECTION_NAME = "wms_schema_free"       # change if your collection name differs

embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

print("✅ Chroma collection ready. Count:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Chroma collection ready. Count: 13


In [ ]:
!pip -q install -U "transformers>=4.40" accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 59.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"   # keep this (usually ungated)

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, use_fast=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    device_map="auto",
    quantization_config=bnb_config,      # ✅ correct way
    torch_dtype=torch.float16,
)

print("✅ LLM loaded with 4-bit:", LLM_NAME)
print("CUDA available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM loaded with 4-bit: Qwen/Qwen2.5-3B-Instruct
CUDA available: True


In [ ]:
import re
import sqlite3
import pandas as pd

DB_NAME = "warehouse.db"   # your sqlite file from earlier

# ----------------------------
# 1) Retrieve schema chunks
# ----------------------------
def retrieve_schema_chunks(question: str, k: int = 4) -> str:
    q_emb = embedder.encode([question], normalize_embeddings=True).tolist()

    res = collection.query(
        query_embeddings=q_emb,
        n_results=k
    )

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]

    # Build a neat schema context block
    lines = []
    for doc, meta in zip(docs, metas):
        table = meta.get("table", "unknown_table") if isinstance(meta, dict) else "unknown_table"
        lines.append(f"- {table}: {doc}")

    return "\n".join(lines)


# ----------------------------
# 2) Build prompt
# ----------------------------
def build_prompt(question: str, retrieved_schema: str) -> str:
    # Instruction is strict: ONLY SQL.
    return f"""You are a SQL expert.
You must write a valid SQLite SELECT query only.
Return ONLY the SQL query (no markdown, no explanation).

Schema context (most relevant tables/chunks):
{retrieved_schema}

User question:
{question}

Rules:
- Use only tables/columns implied by the schema context.
- Prefer order_header/order_detail for order questions.
- If the question asks about a specific order id, filter by order_id.
- Return only one SQL statement.
"""


# ----------------------------
# 3) Generate SQL using local LLM
# ----------------------------
def generate_sql(prompt: str, max_new_tokens: int = 160) -> str:
    # Chat template handling differs by model family.
    # Qwen Instruct supports chat template; fallback to raw prompt if not.
    try:
        messages = [
            {"role": "system", "content": "You are a helpful assistant that outputs only SQL."},
            {"role": "user", "content": prompt},
        ]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    out = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract just the SQL: take last SELECT/WITH block found
    # This helps when the model echoes instructions.
    sql = extract_sql_from_text(out)
    return sql.strip()


def extract_sql_from_text(text: str) -> str:
    # Grab last occurrence of SELECT or WITH and return from there
    m = list(re.finditer(r"\b(SELECT|WITH)\b", text, flags=re.IGNORECASE))
    if not m:
        return text.strip()
    start = m[-1].start()
    candidate = text[start:].strip()

    # Stop at a blank line if the model appended anything extra
    candidate = candidate.split("\n\n")[0].strip()

    # Remove trailing code fences if any
    candidate = candidate.replace("```sql", "").replace("```", "").strip()

    return candidate


# ----------------------------
# 4) Validate SQL (safety)
# ----------------------------
BANNED = [
    "DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE",
    "ATTACH", "DETACH", "PRAGMA", "REINDEX", "VACUUM", "CREATE"
]

def validate_sql(sql: str) -> tuple[bool, str]:
    if not sql or not isinstance(sql, str):
        return False, "Empty SQL."

    s = sql.strip()

    # Must start with SELECT (or WITH if you allow CTEs)
    if not re.match(r"^(SELECT|WITH)\b", s, flags=re.IGNORECASE):
        return False, "Rejected: SQL must start with SELECT or WITH."

    # Reject multi-statement (very basic)
    if ";" in s.strip().rstrip(";"):
        return False, "Rejected: Multiple statements detected."

    upper = s.upper()
    for bad in BANNED:
        if re.search(rf"\b{bad}\b", upper):
            return False, f"Rejected: forbidden keyword '{bad}'."

    return True, "OK"


# ----------------------------
# 5) Execute SQL and return dataframe
# ----------------------------
def run_sql(sql: str) -> pd.DataFrame:
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON;")
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df


# ----------------------------
# 6) Full Text-to-SQL chain with 1 retry on error
# ----------------------------
def text_to_sql(question: str, k: int = 4, retry: bool = True):
    retrieved_schema = retrieve_schema_chunks(question, k=k)
    prompt = build_prompt(question, retrieved_schema)
    sql = generate_sql(prompt)

    ok, msg = validate_sql(sql)
    if not ok:
        return {"question": question, "sql": sql, "status": "REJECTED", "reason": msg, "df": None}

    try:
        df = run_sql(sql)
        return {"question": question, "sql": sql, "status": "OK", "reason": "Executed", "df": df}

    except Exception as e:
        if not retry:
            return {"question": question, "sql": sql, "status": "ERROR", "reason": str(e), "df": None}

        # One repair attempt: include error text + schema again
        repair_prompt = prompt + f"\n\nThe SQL you wrote caused this SQLite error:\n{str(e)}\nFix the SQL. Return ONLY the corrected SQL."
        sql2 = generate_sql(repair_prompt)

        ok2, msg2 = validate_sql(sql2)
        if not ok2:
            return {"question": question, "sql": sql2, "status": "REJECTED", "reason": msg2, "df": None}

        try:
            df2 = run_sql(sql2)
            return {"question": question, "sql": sql2, "status": "OK_AFTER_RETRY", "reason": "Fixed and executed", "df": df2}
        except Exception as e2:
            return {"question": question, "sql": sql2, "status": "ERROR", "reason": str(e2), "df": None}

##Test

In [ ]:
result = text_to_sql("How CASE-0001 is reconciled?")
print("Status:", result["status"])
print("Reason:", result["reason"])
print("SQL:\n", result["sql"])
result["df"].head(20) if result["df"] is not None else None

Status: OK
Reason: Executed
SQL:
 SELECT * FROM case_header WHERE case_id = 'CASE-0001' AND reconciliation_date IS NOT NULL AND reconciliation_date IS NOT NULL AND reconciliation_date < CASE_HEADER.updated_ts


,case_id,container_id,status,expected_date,case_received_date,reconciliation_due_date,reconciliation_date,reconciled_by,created_ts,updated_ts
0,CASE-0001,CONT-13,RECONCILED,2025-10-29,2025-10-30,2025-11-06,2025-11-06,SYSTEM,2025-10-24 19:14:56,2025-11-06 17:04:42
